In [1]:
import os

In [5]:
from models import GPTAssistant

#### post processing utils

In [38]:
def _tokens_equivalent(tok1: str, tok2: str) -> bool:
        """
        Check if two tokens are equivalent.
        
        Tokens are equivalent if:
        1. They are exactly equal (tok1 == tok2)
        2. Both are pure whitespace (any combination of spaces, newlines, tabs, nbsp, etc.)
        3. Both are manual_label tags (opening or closing)
        
        Examples:
        - 'hello' == 'hello' -> True
        - 'hello' == 'world' -> False
        - ' ' == '\n\xa0\n\n\n\n' -> True (both are pure whitespace)
        - ',' == ',' -> True
        - ',' == ' ' -> False
        - '<manual_label labelname="mention">' == '<manual_label labelname="title">' -> True
        """
        # First check: exact equality
        if tok1 == tok2:
            return True
        
        # Second check: both are pure whitespace (different variants)
        if not tok1.strip() and not tok2.strip():
            return True
        
        # Third check: both are manual_label tags (opening or closing)
        
        if is_manual_label_tag(tok1) != 0 and is_manual_label_tag(tok2) != 0:
            return True
        
        # Otherwise, not equivalent
        return False


def merge_tokens_with_auto_labels(tokens: list[str], processed_tokens: list[str], output_dir=None, filename=None, log=False) -> list[str]:
    """
    Merge original tokens with processed tokens to produce the original text
    with only <auto_label ...> tags inserted from processed_tokens.

    Algorithm:
    - If tokens are equivalent (same or both whitespace): take original token, advance both indices
    - If tokens differ:
      - If processed token is a auto_label tag: it's an insertion, take it and advance idx2 only
      - Otherwise: take original token and advance idx1 only
    
    This handles insertions of <auto_label> tags in the processed version.
    """
    n1 = len(tokens)
    n2 = len(processed_tokens)
    result = []
    idx1 = 0
    idx2 = 0
    count = 0

    txt = ""

    while idx1 < n1 and idx2 < n2:
        t1 = tokens[idx1]
        t2 = processed_tokens[idx2]
        txt += f"TOKEN1 {t1}\n"
        txt += f"TOKEN2 {t2}\n"
        
        if _tokens_equivalent(t1, t2):
            # Tokens match: keep original and advance both
            result.append(t1)
            idx1 += 1
            idx2 += 1
            txt += f"COUNT {count}\n"
            
            count = 0
        else:
            count +=1
            # Tokens differ
            if is_auto_label_tag(t2) != 0: # meaning opening or closing auto tag

                if is_auto_label_tag(t2) == 1 : # Meaning opening auto tag
                    # Find the first non-tag word after the opening auto_label tag
                    next_idx = idx2 + 1
                    target_word = None
                    
                    # Look for the first non-tag token after the auto_label opening tag
                    while next_idx < n2:
                        next_token = processed_tokens[next_idx]
                        if not is_tag_token(next_token):
                            target_word = next_token
                            break
                        next_idx += 1
                    
                    if target_word is not None:
                        # Search forward in list1 to find the target word
                        search_idx = idx1
                        found = False
                        
                        while search_idx < n1:
                            if _tokens_equivalent(tokens[search_idx], target_word):
                                # Found the target word in list1
                                # Add all tokens from idx1 to search_idx (excluding search_idx)
                                while idx1 < search_idx:
                                    result.append(tokens[idx1])
                                    txt += f"PRE-TAG TOKEN: {tokens[idx1]}\n"
                                    idx1 += 1
                                
                                # Now add the opening auto_label tag
                                result.append(t2)
                                txt += f"OPENING AUTO_LABEL: {t2}\n"
                                idx2 += 1
                                
                                found = True
                                break
                            search_idx += 1
                        
                        if not found:
                            # If target word not found in list1, just add the auto_label tag
                            result.append(t2)
                            txt += f"AUTO_LABEL (no match found): {t2}\n"
                            idx2 += 1
                    else:
                        # No non-tag token found after auto_label, just add it
                        result.append(t2)
                        txt += f"AUTO_LABEL (no target): {t2}\n"
                        idx2 += 1
                        
                else : # Closing auto tag
                # t2 is a closing auto_label tag (insertion in processed version)
                    result.append(t2)
                    idx2 += 1
            else:
                # t2 is not a auto_label: keep original token
                result.append(t1)
                idx1 += 1
        txt += f"DECISION : {result[-1]}\n"

    # Append remaining tokens from original (if any)
    if idx1 < n1:
        txt += f"t1 {len(tokens[idx1:])}\n"
        result.extend(tokens[idx1:])
    
    # Append remaining tokens from processed (if any, likely closing tags)
    if idx2 < n2:
        txt += f"idx2 {idx2}\n"
        txt += f"{processed_tokens[idx2:]}\n"
        txt += f"t2 {len(processed_tokens[idx2:])}\n"
        result.extend(processed_tokens[idx2:])
    
    if output_dir and filename:
        with open(f"{output_dir}/debug_merge_{filename}.txt", "w", encoding="utf-8") as f:
            f.write(txt)

    if log:
        print(f"   ✓ Merged {len(tokens)} original and {len(processed_tokens)} processed into {len(result)} tokens")
    return result




def add_style_and_parent_to_auto_labels(html_content: str, label_scheme_path: str = None) -> str:
    """
    Add parent and style attributes to auto_label tags in HTML content based on label scheme JSON.
    
    The function loads the label scheme from a JSON file to determine:
    - Parent relationships (top-level labels have parent="", sublabels have parent="<parent_label>")
    - Colors for each label (converted to background-color style)
    
    Args:
        html_content: HTML string with auto_label tags
        label_scheme_path: Path to label_scheme.json file. If None, uses default path.
    
    Returns:
        Modified HTML string with parent and style attributes added
    """
    import re
    import json
    import os
    
    # Default path to label scheme
    if label_scheme_path is None:
        current_dir = f"C:\\Users\\zakga\\OneDrive\\Documents\\code\\labelstudio\\annotation\\llm_based_annotation\\utils\\"
        label_scheme_path = os.path.join(current_dir, '..', '..', 'ressources', 'label_scheme.json')
    
    # Load label scheme
    try:
        with open(label_scheme_path, 'r', encoding='utf-8') as f:
            label_scheme = json.load(f)
    except FileNotFoundError:
        print(f"Warning: Label scheme file not found at {label_scheme_path}. Using auto_label tags without modification.")
        return html_content
    
    # Build parent and style mappings from label scheme
    parent_map = {}
    style_map = {}
    
    # Helper function to determine text color based on background brightness
    def get_text_color(hex_color: str) -> str:
        """Determine if text should be black or white based on background color brightness."""
        # Remove # if present
        hex_color = hex_color.lstrip('#')
        # Convert to RGB
        r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
        # Calculate relative luminance
        luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
        return 'black' if luminance > 0.5 else 'white'
    
    # Convert hex color to rgb format
    def hex_to_rgb(hex_color: str) -> str:
        """Convert hex color to rgb() format."""
        hex_color = hex_color.lstrip('#')
        r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
        return f"rgb({r}, {g}, {b})"
    
    # Process label scheme to build mappings
    for parent_label, parent_data in label_scheme.items():
        # Top-level labels have no parent
        parent_map[parent_label] = ""
        
        # Set style for top-level label
        if 'color' in parent_data:
            bg_color = hex_to_rgb(parent_data['color'])
            text_color = get_text_color(parent_data['color'])
            style_map[parent_label] = f"background-color: {bg_color}; color: {text_color};"
        
        # Process sublabels
        if 'sublabels' in parent_data:
            for sublabel, sublabel_data in parent_data['sublabels'].items():
                # Sublabels have the top-level label as parent
                parent_map[sublabel] = parent_label
                
                # Set style for sublabel
                if 'color' in sublabel_data:
                    bg_color = hex_to_rgb(sublabel_data['color'])
                    text_color = get_text_color(sublabel_data['color'])
                    style_map[sublabel] = f"background-color: {bg_color}; color: {text_color};"
    
    def process_auto_label(match):
        """Process each auto_label opening tag match."""
        full_tag = match.group(0)
        
        # Extract labelname
        labelname_match = re.search(r'labelname="([^"]*)"', full_tag)
        if not labelname_match:
            return full_tag  # No labelname found, return unchanged
        
        labelname = labelname_match.group(1)
        
        # Determine parent attribute from label scheme
        parent_value = parent_map.get(labelname, "")
        parent_attr = f'parent="{parent_value}"'
        
        # Get style attribute from label scheme
        style = style_map.get(labelname, '')
        if style:
            style_attr = f'style="{style}"'
        else:
            style_attr = ''
        
        # Check if parent or style already exist
        has_parent = 'parent=' in full_tag
        has_style = 'style=' in full_tag
        
        # Build new tag
        # Remove existing parent/style if present
        if has_parent:
            full_tag = re.sub(r'\s*parent="[^"]*"', '', full_tag)
        if has_style:
            full_tag = re.sub(r'\s*style="[^"]*"', '', full_tag)
        
        # Insert new attributes before the closing >
        new_tag = full_tag[:-1]  # Remove closing >
        new_tag += f' {parent_attr} {style_attr}>'
        
        return new_tag
    
    # Pattern to match auto_label opening tags
    pattern = r'<auto_label[^>]*>'
    
    # Process all auto_label opening tags
    modified_html = re.sub(pattern, process_auto_label, html_content, flags=re.IGNORECASE)
    
    return modified_html



def compare_html_allow_auto_labels(merged_html: str, original_html: str) -> bool:
    """
    Compare two HTML strings character-by-character, considering them equivalent
    if the only differences are the presence or placement of <auto_label ...>
    and </auto_label> tags.

    Returns True when equal after stripping auto_label tags, else prints
    a concise diff context and returns False.
    """
    a = strip_auto_labels(merged_html)
    b = strip_auto_labels(original_html)
    if a == b:
        print("   ✓ HTMLs match when ignoring auto_label tags")
        return True
    # Find first index of difference
    min_len = min(len(a), len(b))
    diff_idx = None
    for i in range(min_len):
        if a[i] != b[i]:
            diff_idx = i
            print(f"   ✗ Difference at index {diff_idx}: '{a[i]}' vs '{b[i]}'")
            break
    if diff_idx is None and len(a) != len(b):
        diff_idx = min_len
    # Print a small window around the difference
    if diff_idx is not None:
        start = max(0, diff_idx - 50)
        end_a = min(len(a), diff_idx + 50)
        end_b = min(len(b), diff_idx + 50)
        print("   ✗ Difference found (ignoring auto_label):")
        print("--- merged_html (stripped) ---")
        print(a[start:end_a])
        print("--- original_html (stripped) ---")
        print(b[start:end_b])
    else:
        print("   ✗ Difference detected but could not locate index")
    return False

#### html utils


In [11]:
from bs4 import BeautifulSoup
import itertools
import re
    
def extract_body(html_content: str) -> str:
    """
    Extract only the body content from HTML, excluding style, script, and head tags.
    Returns the exact string representation of the <body> element to keep reversibility.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    body = soup.find('body')
    if body is not None:
        return str(body)
    print("   ⚠ Warning: No <body> tag found, returning original content")
    return html_content

def remove_bookmarks(tokens: list) -> list:
    """Remove all 
      - <htmllabelizer_bookmark ...> tokens 
      - </htmllabelizer_bookmark ...> closing tokens 
      - tokens in between 
    from the token list."""
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        if tok.startswith('<htmllabelizer_bookmark'):
            # Find closing tag
            j = i + 1
            while j < len(tokens):
                if tokens[j].startswith('</htmllabelizer_bookmark'):
                    break
                j += 1
            # Remove from i to j (inclusive)
            del tokens[i:j+1]
            continue
        i += 1

    return [tok for tok in tokens if not tok.startswith('<htmllabelizer_bookmark')]

def is_tag_token(tok: str) -> bool:
    """Return True if token looks like an HTML tag (e.g., <...>)."""
    return len(tok) >= 3 and tok[0] == '<' and tok[-1] == '>'


def is_auto_label_tag(tok: str) -> bool:
    """Return 1 if token is an opening, 2 if it is a closing auto_label tag, 0 otherwise"""
    if not is_tag_token(tok):
        return 0
    # Accept variations with attributes on opening tag
    if tok.lower().startswith('<auto_label'):
        return 1
    if tok.lower().startswith('</auto_label'):
        return 2
    return 0

def is_manual_label_tag(tok):
    """Return 1 if token is an opening, 2 if it is a closing manual_label tag, 0 otherwise"""
    if not is_tag_token(tok):
        return 0
    # Accept variations with attributes on opening tag
    if tok.lower().startswith('<manual_label'):
        return 1
    if tok.lower().startswith('</manual_label'):
        return 2
    return 0

def strip_auto_labels(html: str) -> str:
    """Remove all <auto_label ...> and </auto_label> tags from html."""
    # Remove opening tags with any attributes
    html_no_open = re.sub(r"<\s*auto_label\b[^>]*>", "", html, flags=re.IGNORECASE)
    # Remove closing tags
    html_no_tags = re.sub(r"<\s*/\s*auto_label\s*>", "", html_no_open, flags=re.IGNORECASE)
    return html_no_tags







#### html class

In [12]:
class HTMLLabel:
    """
    Parse and represent manual_label or auto_label HTML tokens.
    
    Example tokens:
        <manual_label labelname="Authority_Mention" jurisdiction="Canada">
        <auto_label labelname="Legal_Issue" confidence="0.95">
    """
    
    def __init__(self, token: str):
        """
        Initialize HTMLLabel from a token string.
        
        Args:
            token: String token like '<manual_label labelname="xyz" attr="val">'
        
        Raises:
            ValueError: If token is not a valid manual_label or auto_label tag
        """
        if not self._is_valid_label_token(token):
            raise ValueError(f"Token is not a valid manual_label or auto_label: {token}")
        
        self._token = token
        self._label_type = self._detect_label_type(token)
        self._attributes = self._parse_attributes(token)
        
        if 'labelname' not in self._attributes:
            raise ValueError(f"Token missing 'labelname' attribute: {token}")
    
    def _is_valid_label_token(self, token: str) -> bool:
        """Check if token is a valid manual_label or auto_label opening tag."""
        if not token.startswith('<') or not token.endswith('>'):
            return False
        lower = token.lower()
        return lower.startswith('<manual_label') or lower.startswith('<auto_label')
    
    def _detect_label_type(self, token: str) -> str:
        """Detect whether token is 'manual_label' or 'auto_label'."""
        if token.lower().startswith('<manual_label'):
            return 'manual_label'
        elif token.lower().startswith('<auto_label'):
            return 'auto_label'
        return None
    
    def _parse_attributes(self, token: str) -> dict:
        """Parse all attributes from the tag into a dictionary."""
        # Remove < and > brackets
        inner = token[1:-1]
        
        # Remove tag name (manual_label or auto_label)
        if inner.lower().startswith('manual_label'):
            inner = inner[12:].strip()
        elif inner.lower().startswith('auto_label'):
            inner = inner[10:].strip()
        
        # Parse attributes using regex
        # Matches: attr="value" or attr='value'
        attr_pattern = re.compile(r'(\w+)\s*=\s*["\']([^"\']*)["\']')
        attributes = {}
        
        for match in attr_pattern.finditer(inner):
            key, value = match.groups()
            attributes[key] = value
        
        return attributes
    
    def _update_from_token(self, new_token: str):
        """
        Update internal state by re-parsing the new token.
        
        Args:
            new_token: New token string to parse
        """
        self._token = new_token
        self._label_type = self._detect_label_type(new_token)
        self._attributes = self._parse_attributes(new_token)
    
    def is_manual_label(self) -> bool:
        """Return True if this is a manual_label."""
        return self._label_type == 'manual_label'
    
    def is_auto_label(self) -> bool:
        """Return True if this is an auto_label."""
        return self._label_type == 'auto_label'
    
    @property
    def name(self) -> str:
        """Return the labelname attribute value."""
        return self._attributes.get('labelname', '')
    
    @property
    def attributes(self) -> dict:
        """Return dictionary of all attributes (including labelname)."""
        return self._attributes.copy()
    
    @property
    def label_type(self) -> str:
        """Return 'manual_label' or 'auto_label'."""
        return self._label_type
    
    def to_string(self, remove_attributes: list = None, keep_attributes: list = None):
        """
        Update the token with filtered attributes and update internal state.
        
        Args:
            remove_attributes: List of attribute names to remove. If None, no removal filtering.
            keep_attributes: List of attribute names to keep (all others removed). If None, no keep filtering.
        
        Note: Only one of remove_attributes or keep_attributes should be provided, not both.
              If both are provided, keep_attributes takes precedence.
              This method modifies the object's internal state.
        
        Examples:
            >>> label = HTMLLabel('<manual_label labelname="mention" docid="123" style="color:red" parent="div">')
            
            # Remove specific attributes (modifies the object)
            >>> label.to_string(remove_attributes=['style', 'parent'])
            >>> print(label)  # '<manual_label labelname="mention" docid="123">'
            
            # Keep only specific attributes (modifies the object)
            >>> label.to_string(keep_attributes=['labelname'])
            >>> print(label)  # '<manual_label labelname="mention">'
        """
        # If both provided, keep_attributes takes precedence
        if keep_attributes is not None:
            # Keep only specified attributes
            filtered_attrs = {k: v for k, v in self._attributes.items() if k in keep_attributes}
        elif remove_attributes is not None:
            # Remove specified attributes
            filtered_attrs = {k: v for k, v in self._attributes.items() if k not in remove_attributes}
        else:
            # No filtering, do nothing
            return
        
        # Reconstruct token
        tag_name = self._label_type
        reconstructed = f'<{tag_name}'
        
        # Add remaining attributes
        for key, value in filtered_attrs.items():
            reconstructed += f' {key}="{value}"'
        
        reconstructed += '>'
        
        # Update internal state
        self._update_from_token(reconstructed)
    
    def switch_type(self):
        """
        Switch between manual_label and auto_label types and update internal state.
        
        If the label is manual_label, it becomes auto_label, and vice versa.
        All attributes are preserved. This method modifies the object's internal state.
        
        Examples:
            >>> label = HTMLLabel('<manual_label labelname="mention" docid="123">')
            >>> label.switch_type()
            >>> print(label)  # '<auto_label labelname="mention" docid="123">'
            >>> label.is_auto_label()  # True
            
            >>> label.switch_type()  # Switch back
            >>> print(label)  # '<manual_label labelname="mention" docid="123">'
            >>> label.is_manual_label()  # True
        """
        # Determine the new label type
        new_type = 'auto_label' if self._label_type == 'manual_label' else 'manual_label'
        
        # Reconstruct token with new type
        reconstructed = f'<{new_type}'
        
        # Add all attributes
        for key, value in self._attributes.items():
            reconstructed += f' {key}="{value}"'
        
        reconstructed += '>'
        
        # Update internal state
        self._update_from_token(reconstructed)
    
    def to_simplified(self) -> str:
        """
        Return simplified token format using labelname as tag name, preserving other attributes.
        
        Converts:
            <manual_label labelname="mention" docid="123"> → <mention docid="123">
            <auto_label labelname="title" titletype="main"> → <title titletype="main">
        
        This method removes the manual_label/auto_label wrapper and the labelname attribute,
        but keeps all other attributes. It does NOT modify the object's internal state.
        
        Returns:
            Simplified token string with format <labelname attr="value" ...>
        
        Examples:
            >>> label = HTMLLabel('<manual_label labelname="mention" docid="123" style="color:red">')
            >>> label.to_simplified()
            '<mention docid="123" style="color:red">'
            
            >>> label2 = HTMLLabel('<auto_label labelname="title" titletype="main">')
            >>> label2.to_simplified()
            '<title titletype="main">'
            
            >>> label3 = HTMLLabel('<manual_label labelname="decision">')
            >>> label3.to_simplified()
            '<decision>'
            
            >>> print(label)  # Original token unchanged
            '<manual_label labelname="mention" docid="123" style="color:red">'
        """
        labelname = self._attributes.get('labelname', '')
        
        # Start with the labelname as tag
        simplified = f'<{labelname}'
        
        # Add all attributes except 'labelname'
        for key, value in self._attributes.items():
            if key != 'labelname':
                simplified += f' {key}="{value}"'
        
        simplified += '>'
        return simplified
    
    def __repr__(self):
        return f"HTMLLabel(type={self._label_type}, name={self.name}, attrs={self.attributes})"
    
    def __str__(self):
        return self._token

In [13]:
def from_simplified(simplified_token: str, label_type: str = 'auto_label') -> HTMLLabel:
    """
    Convert a simplified token format to a full HTMLLabel object.
    
    Takes a simplified token like <title titletype="main"> and converts it to
    a full label format like <auto_label labelname="title" titletype="main">.
    
    Args:
        simplified_token: Simplified token string like '<title titletype="main">'
        label_type: Either 'manual_label' or 'auto_label' (default: 'auto_label')
    
    Returns:
        HTMLLabel object with the full token format
    
    Raises:
        ValueError: If simplified_token is not a valid tag or label_type is invalid
    
    Examples:
        >>> label = from_simplified('<title titletype="main">', 'manual_label')
        >>> print(label)
        '<manual_label labelname="title" titletype="main">'
        
        >>> label2 = from_simplified('<mention docid="123">', 'auto_label')
        >>> print(label2)
        '<auto_label labelname="mention" docid="123">'
        
        >>> label3 = from_simplified('<decision>')
        >>> print(label3)
        '<auto_label labelname="decision">'
    """
    # Validate label_type
    if label_type not in ['manual_label', 'auto_label']:
        raise ValueError(f"label_type must be 'manual_label' or 'auto_label', got: {label_type}")
    
    # Validate simplified_token format
    if not simplified_token.startswith('<') or not simplified_token.endswith('>'):
        raise ValueError(f"Invalid token format: {simplified_token}")
    
    # Remove < and > brackets
    inner = simplified_token[1:-1].strip()
    
    # Parse tag name and attributes
    # Split on first space to separate tag name from attributes
    tokens = inner.split()

    name_tokens = []
    attr_tokens = []

    for tok in tokens:
        if '=' in tok:
            attr_tokens.append(tok)
        else:
            if not attr_tokens:
                name_tokens.append(tok)
            else:
                # Edge case: malformed token after attributes
                attr_tokens.append(tok)

    tag_name = ' '.join(name_tokens)
    other_attrs = ' '.join(attr_tokens)

    
    # Construct full token
    full_token = f'<{label_type} labelname="{tag_name}"'
    
    if other_attrs:
        full_token += f' {other_attrs}'
    
    full_token += '>'
    
    # Return HTMLLabel object
    return HTMLLabel(full_token)

#### tokenizer utils

In [16]:
import re

def tokenize(html_body: str, print=False) -> list:
    """
    Split HTML into a sequence of tokens that preserves:
    - HTML tags as single tokens (e.g., '<div class="x">')
    - Whitespace runs as separate tokens (spaces, newlines, tabs)
    - Punctuation as separate tokens (e.g., ',', '.', ';')
    - Words as separate tokens
    This ensures decode(tokens) == html_body with simple join AND prevents
    merging of words with punctuation after tag removal.
    """
    # Enhanced pattern: separates tags, whitespace, punctuation, words, and other chars
    # Group 1: HTML tags
    # Group 2: Whitespace runs
    # Group 3: Common punctuation (as separate tokens)
    # Group 4: Word characters (alphanumeric + underscore)
    # Group 5: Any other single character
    pattern = re.compile(r"(<[^>]*>)|(\s+)|([.,;:!?()[\]{}\"'`‑–—])|(\w+)|([^\w\s<>])")
    tokens = []
    for m in pattern.finditer(html_body):
        tok = m.group(1) or m.group(2) or m.group(3) or m.group(4) or m.group(5)
        if tok:  # Safety check
            tokens.append(tok)

    if print:
        print(f"   ✓ Tokenized into {len(tokens)} tokens (tags+whitespace+punctuation+words, reversible)")
    return tokens


def decode(tokens: list, print=False) -> str:
    """
    Reconstruct HTML body by concatenating tokens exactly.
    """
    reconstructed_html = "".join(tokens)

    if print:
        print(f"   ✓ Decoded {len(tokens)} tokens into HTML")
    return reconstructed_html

#### cleaner utils

In [17]:
def _strip_tags(tokens: list, keep_manual_label=False, keep_auto_label=False, keep_bookmarks=False, merge = True, log=False) -> list:
    """
    Remove tag tokens from the HTML token sequence and return text token list
    while preserving whitespace tokens exactly.
    - Drops any token that starts with '<' and ends with '>' (except manual_label tags if keep_manual_label=True)
    - Keeps whitespace and text tokens unchanged
    - If keep_manual_label=True: keeps all <manual_label ...> and </manual_label> tags
    """

    # --- REMOVE TAG TOKENS ----
    text_parts = []
    for t in tokens:
        if len(t) >= 2 and t[0] == '<' and t[-1] == '>':
            # Check if we should keep manual_label tags
            if keep_manual_label:
                # Keep manual_label tags (opening and closing)
                if t.lower().startswith('<manual_label') or t.lower().startswith('</manual_label'):
                    text_parts.append(t)
                    continue

            if keep_auto_label:
                # Keep auto_label tags (opening and closing)
                if t.lower().startswith('<auto_label') or t.lower().startswith('</auto_label'):
                    text_parts.append(t)
                    continue

            if keep_bookmarks:
                # Keep bookmarks tags (opening and closing)
                if t.lower().startswith('<htmllabelizer_bookmark') or t.lower().startswith('</htmllabelizer_bookmark>'):
                    text_parts.append(t)
                    continue
            # Skip all other tags
            continue
        text_parts.append(t)

    # --- MERGE WHITESPACE TOKENS ---- 
    if merge:
        merge_tokens = []
        accumulated_whitespace = ""
        
        for tok in text_parts:
            # Check if token is pure whitespace
            if tok and not tok.strip():
                # Accumulate whitespace
                accumulated_whitespace += tok
            else:
                # Non-whitespace token found
                # First, flush any accumulated whitespace
                if accumulated_whitespace:
                    merge_tokens.append(accumulated_whitespace)
                    accumulated_whitespace = ""
                # Then add the current non-whitespace token
                merge_tokens.append(tok)
        
        # Don't forget any trailing whitespace
        if accumulated_whitespace:
            merge_tokens.append(accumulated_whitespace)

    if log:
        print(f"   ✓ Stripped tags: {len(tokens)} -> text token length {len(text_parts)}")
    return text_parts if not merge else merge_tokens


def _normalize_text_tokens(text_tokens: list, log=False) -> list:
    """
    Normalize complex whitespace tokens by replacing them with a single space.
    
    Rules:
    - Tokens with text content (letters, numbers, punctuation): KEEP AS-IS
    - Simple whitespace tokens (' ', '\n', '\t'): KEEP AS-IS
    - Complex whitespace tokens ('\n\n\n\xa0\xa0\n\n\xa0\n'): REPLACE with ' '
    
    A complex whitespace token = pure whitespace with length > 1
    """
    normalized = []
    
    for tok in text_tokens:
        # If token has any non-whitespace character, keep as-is
        if tok.strip():
            normalized.append(tok)
        # Token is pure whitespace
        elif len(tok) == 1:
            # Simple single-character whitespace: keep as-is
            normalized.append(tok)
        else:
            # Complex multi-character whitespace: replace with single space
            normalized.append(' ')

    if log:
        print(f"   ✓ Normalized text tokens: {len(text_tokens)} -> {len(normalized)}")

    return normalized


def clean_tokens(html_tokens: list, normalize: bool = False, keep_manual_label=False, keep_auto_label=False, keep_bookmarks=False, log=False) -> list:
    """
    Produce a cleaned token list by:
    1) Removing all tag tokens and building plain text token list
    2) Optionally normalizing whitespace tokens (NBSPs and excessive newlines/spaces)
    Returns: cleaned tokens (no tag tokens)
    """
    text_tokens = _strip_tags(tokens=html_tokens, keep_manual_label=keep_manual_label, keep_auto_label=keep_auto_label, keep_bookmarks=keep_bookmarks, log=log)
    if normalize:
        text_tokens = _normalize_text_tokens(text_tokens, log=log)

    if log:
        print(f"   ✓ Cleaned tokens count: {len(text_tokens)} (normalize={normalize})")
    return text_tokens

#### chunker utils

In [18]:
import itertools
import re 

def chunk_tokens(tokens: list, min_tokens: int = 500, stop_bookmark_separation=False) -> list:
    """
    Chunk a list of tokens into chunks of at least min_tokens.
    
    STRICT RULES:
    1. Accumulate tokens until reaching min_tokens
    2. After min_tokens, continue ONLY until:
        - Quotations are closed (not inside_quotes)
        - AND <manual_label> tags are closed (depth == 0)
        - AND current token is a word (not whitespace/punctuation)
    
    If stop_bookmark_separation=True, splits at bookmark first, then chunks each part separately.
    
    Returns list of token chunks, or tuple of two lists if bookmark separation.
    """
    
    # If stop_bookmark_separation is True, find and split at bookmark
    if stop_bookmark_separation:
        bookmark_idx = None
        for i in range(len(tokens) - 2):
            if (tokens[i] == '<htmllabelizer_bookmark id="stop">' and 
                tokens[i+1] == '🔖' and 
                tokens[i+2] == '</htmllabelizer_bookmark>'):
                bookmark_idx = i
                break
        
        if bookmark_idx is not None:
            # Split tokens into before and after (excluding the 3 bookmark tokens)
            tokens_before = tokens[:bookmark_idx]
            tokens_after = tokens[bookmark_idx + 3:]
            
            print(f"   ✓ Found bookmark separator at index {bookmark_idx}")
            print(f"   ✓ Splitting: {len(tokens_before)} tokens before, {len(tokens_after)} tokens after")
            
            # Chunk both parts separately
            chunks_before = _chunk_token_list(tokens_before, min_tokens)
            chunks_after = _chunk_token_list(tokens_after, min_tokens)
            
            print(f"   ✓ Total chunks: {len(chunks_before)} before + {len(chunks_after)} after = {len(chunks_before) + len(chunks_after)}")
            return chunks_before, chunks_after
        else:
            print(f"   ⚠ Warning: stop_bookmark_separation=True but bookmark not found")
    
    # Default behavior: chunk entire list
    return _chunk_token_list(tokens, min_tokens)

def _chunk_token_list(tokens: list, min_tokens: int) -> list:
    """
    Internal method to chunk a token list.
    
    Algorithm:
    1. Accumulate tokens until count >= min_tokens
    2. After min_tokens, continue until ALL conditions are met:
        - NOT inside quotes
        - manual_label depth == 0
        - current token is a word
    """
    chunks = []
    current = []
    count = 0
    
    inside_quotes = False
    manual_label_depth = 0
    
    def is_word_token(tok: str) -> bool:
        """Check if token is a word (alphanumeric, >=1 char)."""
        return tok and tok.strip() and re.match(r'^\w+$', tok) is not None
    
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        current.append(tok)
        count += 1
        
        # Track quote state - ONLY opening quote " starts, ONLY closing quote " ends
        if tok == '“':  # U+201C - LEFT DOUBLE QUOTATION MARK (opening)
            inside_quotes = True
        elif tok == '”':  # U+201D - RIGHT DOUBLE QUOTATION MARK (closing)
            inside_quotes = False
        
        # Track manual_label tag depth
        if tok.startswith('<manual_label'):
            manual_label_depth += 1
        elif tok.startswith('</manual_label'):
            manual_label_depth -= 1
        
        # Check if we can end the chunk
        if count >= min_tokens:
            # Check ALL conditions:
            # 1. Not inside quotes
            # 2. manual_label depth is 0
            # 3. Current token is a word
            if not inside_quotes and manual_label_depth == 0 and is_word_token(tok):
                # All conditions met: end chunk here
                chunks.append(current)
                current = []
                count = 0
                # Reset states for safety
                inside_quotes = False
                manual_label_depth = 0
        
        i += 1
    
    # Append remaining tokens as final chunk
    if current:
        chunks.append(current)
    
    print(f"   ✓ Chunked tokens into {len(chunks)} chunks (>= {min_tokens} tokens each)")
    return chunks




def flatten_token_chunks(token_chunks: list[list[str]]) -> list[str]:
    """
    Flatten a list of token chunks (list of lists) back into a single token list.
    Preserves token order exactly.
    """
    flat = list(itertools.chain.from_iterable(token_chunks))
    print(f"   ✓ Flattened {len(token_chunks)} chunks into {len(flat)} tokens")
    return flat

#### few shot utils

In [19]:
def _prepare_label_tokens(chunk, label_config):
    """
    Transform label tokens according to specified parameters.
    
    Handles both opening and closing tags, using a stack to track label names
    for proper closing tag generation in simplified mode.
    
    Args:
        chunk: List of tokens to process
        remove_attributes: List of attribute names to remove from labels
        keep_attributes: List of attribute names to keep in labels
        switch_type: If True, switch between manual_label and auto_label types
        use_simplified: If True, output simplified form (e.g., <title> instead of <manual_label labelname="title">)
        keep_labels: List of label names to keep (all others removed). If None, keep all labels.
        remove_labels: List of label names to remove. If None, no removal filtering.
    
    Returns:
        List of transformed tokens
    """

    # Get config parameters
    remove_attributes = label_config.get('remove_attributes', None)
    keep_attributes = label_config.get('keep_attributes', None)
    switch_type = label_config.get('switch_type', False)
    use_simplified = label_config.get('use_simplified', False)
    keep_labels = label_config.get('keep_labels', None)
    remove_labels = label_config.get('remove_labels', None)


    transformed_chunk = []
    label_name_stack = []  # Stack to track opened label names for closing tags
    skip_stack = []  # Stack to track which labels are being skipped (removed)
    
    for token in chunk:
        # Check if token is a label opening tag
        is_manual_open = token.lower().startswith('<manual_label') and token.endswith('>')
        is_auto_open = token.lower().startswith('<auto_label') and token.endswith('>')
        
        # Check if token is a label closing tag
        is_manual_close = token.lower().startswith('</manual_label') and token.endswith('>')
        is_auto_close = token.lower().startswith('</auto_label') and token.endswith('>')
        
        if is_manual_open or is_auto_open:
            # Process opening tag
            try:
                label = HTMLLabel(token)
                
                # Check if this label should be kept based on keep_labels/remove_labels
                should_keep = True
                if keep_labels is not None:
                    should_keep = label.name in keep_labels
                elif remove_labels is not None:
                    should_keep = label.name not in remove_labels
                
                if not should_keep:
                    # Skip this label tag, but track it for closing tag
                    skip_stack.append(True)
                    # Don't add to transformed_chunk (removes the tag but keeps content)
                    continue
                
                skip_stack.append(False)
                
                # Apply attribute filtering if specified
                if keep_attributes is not None or remove_attributes is not None:
                    label.to_string(remove_attributes=remove_attributes, keep_attributes=keep_attributes)
                
                # Switch type if requested
                if switch_type:
                    label.switch_type()
                
                # Output simplified or full form
                if use_simplified:
                    transformed_token = label.to_simplified()
                    # Push label name to stack for closing tag
                    label_name_stack.append(label.name)
                else:
                    transformed_token = str(label)
                
                transformed_chunk.append(transformed_token)
            except ValueError:
                # If token can't be parsed, keep as-is
                transformed_chunk.append(token)
                
        elif is_manual_close or is_auto_close:
            # Process closing tag
            # Check if we skipped the corresponding opening tag
            if skip_stack:
                was_skipped = skip_stack.pop()
                if was_skipped:
                    # Skip this closing tag too (removes the tag but keeps content)
                    continue
            
            if use_simplified:
                # Pop label name from stack
                if label_name_stack:
                    label_name = label_name_stack.pop()
                    transformed_token = f'</{label_name}>'
                else:
                    # Stack empty, keep as-is (shouldn't happen with valid HTML)
                    transformed_token = token
            else:
                # Not simplified: handle type switching if needed
                if switch_type:
                    # Switch closing tag type
                    if is_manual_close:
                        transformed_token = '</auto_label>'
                    else:
                        transformed_token = '</manual_label>'
                else:
                    transformed_token = token
            
            transformed_chunk.append(transformed_token)
        else:
            # Not a label tag, keep as-is
            transformed_chunk.append(token)
    
    return transformed_chunk


def extract_few_shot_examples(token_chunks, label_config):
    """
    Extract few-shot examples from the chunks with flexible label transformation.
    
    Args:
        token_chunks: List of token chunks to process
        remove_attributes: List of attribute names to remove from labels. If None, no removal filtering.
        keep_attributes: List of attribute names to keep in labels (all others removed). If None, no keep filtering.
        switch_type: If True, switch between manual_label and auto_label types
        use_simplified: If True, output simplified form (e.g., <title> instead of <manual_label labelname="title">)
        keep_labels: List of label names to keep (all others removed). If None, keep all labels.
        remove_labels: List of label names to remove. If None, no removal filtering.
    
    Returns:
        list: List of tuples (input_tokens, output_tokens)
              - input_tokens: cleaned tokens without any label tags
              - output_tokens: tokens with transformed label tags according to parameters
    
    Examples:
        >>> # Remove specific attributes and switch to auto_label
        >>> examples = extract_few_shot_examples(chunks, remove_attributes=['style', 'parent'], switch_type=True)
        
        >>> # Keep only labelname and docid attributes
        >>> examples = extract_few_shot_examples(chunks, keep_attributes=['labelname', 'docid'])
        
        >>> # Output simplified form with proper closing tags
        >>> examples = extract_few_shot_examples(chunks, use_simplified=True)
        >>> # <manual_label labelname="title">Text</manual_label> → <title>Text</title>
        
        >>> # Keep only decision labels (removes title tags but keeps text)
        >>> examples = extract_few_shot_examples(chunks, keep_labels=['decision'], use_simplified=True)
        >>> # <decision><title>John Campbell Law Corporation v.</title></decision>
        >>> # → <decision>John Campbell Law Corporation v.</decision>
        
        >>> # Remove title labels (keeps all other labels)
        >>> examples = extract_few_shot_examples(chunks, remove_labels=['title'], use_simplified=True)
        >>> # Same result as above
    """
    examples = []
    
    for chunk in token_chunks:
        # Input: tokens cleaned from all label tags
        input_chunk = clean_tokens(chunk, normalize=True, keep_manual_label=False, 
                                   keep_auto_label=False, keep_bookmarks=False)
        
        # Output: tokens with transformed label tags
        output_chunk = _prepare_label_tokens(
            chunk, 
            label_config
        )
        
        examples.append((decode(input_chunk), decode(output_chunk)))
    
    print(f"   ✓ Extracted {len(examples)} few-shot examples from chunks")
    return examples


def select_few_shot(examples, n):
    """
    Select n few-shot examples from the provided list.
    
    Args:
        examples: List of tuples (input, expected_output)
        n: Number of examples to select
    Returns:
        list: Selected few-shot examples
    """
    if n >= len(examples):
        return examples
    else:
        return examples[:n]

#### prompt utils


In [20]:
def get_prompt_processing(prompt_path,few_shot_examples=None):
    """
    Create system and user prompts for the AI model to process legal text.
    
    Args:
        few_shot_examples: List of tuples (input, expected_output) for few-shot learning
    
    Returns:
        tuple: (system_prompt, user_prompt_template)
    """
    with open(prompt_path, 'r', encoding='utf-8') as f:
        system_prompt = f.read()
        
    
    # Add few-shot examples if provided
    if few_shot_examples:
        system_prompt += "\n\nHere are some examples:\n"
        for i, (input_text, expected_output) in enumerate(few_shot_examples, 1):
            system_prompt += f"\nExample {i}:\n"
            system_prompt += f"<ORIGINAL_TEXT>{input_text}<END_ORIGINAL_TEXT>\n"
            system_prompt += f"<EXPECTED_OUTPUT>{expected_output}<END_EXPECTED_OUTPUT>\n"
    
    user_prompt_template = """Please annotate the following legal text with the appropriate auto_label tags:

    <ORIGINAL_TEXT>{text}<END_ORIGINAL_TEXT>
    
    OUTPUT:"""
    
    return system_prompt, user_prompt_template

def get_prompt_fallback_consistency():
    """
    Create system and user prompts for the AI model to save consistency errors.
        
    Returns:
        tuple: (system_prompt, user_prompt_template)
    """
    system_prompt = f"""
        You are a legal assistant specialized in automatic annotation of legal texts. Your role is to correct errors in previously annotated texts.

        You will receive :
        1. The Original text.
        2. A version of the text annotated with <tag> tags that contains at least one error of consistency. 
        
        A consistency error means that some oppening <tag> tags or closing </tag> tags are missing. 
        The oppening tag count doesn't match the closing tag count.



        You must carefully detect where the missing <tag> is missing and correct the issue while respecting the annotation schema below.

    """
    
    
    user_prompt_template = """Please correct the annotated text to fix any discrepancies with the original text while adhering to the annotation schema:

    Input: "{text_pair}"
    
    Output:"""
    
    return system_prompt, user_prompt_template

def get_prompt_fallback_hallucination():
    """
    Create system and user prompts for the AI model to save hallucination errors.
        
    Returns:
        tuple: (system_prompt, user_prompt_template)
    """
    system_prompt = """
        You are a legal assistant specialized in automatic annotation of legal texts. Your role is to correct errors in previously annotated texts.

        You will receive:
        1. The original text.
        2. A version of the text annotated with <tag> tags that contains at least one error.

        The upstream error-detection system compares the original text and the annotated text token by token. After removing all <tag> tags, both token lists must match exactly. When they do not, the text pair is sent to you for correction.

        Typical error patterns include:
        - Incorrect annotation boundaries at the beginning or end of a labeled span.
        - Added or removed quotation marks.
        - Added or removed spaces or line breaks.

        You must carefully correct these issues while respecting the annotation schema below.


        Follow these guidelines meticulously to ensure accurate and consistent legal text annotations.
    """
    
    
    user_prompt_template = """Please correct the annotated text to fix any discrepancies with the original text while adhering to the annotation schema:

    Input: "{text_pair}"
    
    Output:"""
    
    return system_prompt, user_prompt_template

#### verification utils

In [21]:
"""
Verification utilities for validating LLM-generated annotated text.
These functions check the validity of processed chunks without modifying them.
"""

import json
from typing import Dict, List, Tuple, Optional
from bs4 import BeautifulSoup


# Label scheme definition based on label schemes.html
LABEL_SCHEME = {
    "legislation": {
        "attributes": ["docid", "uri"],  # title, citation, fragment
        "required": False
    },
    "decision": {
        "attributes": ["docid", "uri"],  # title, citation, fragment
        "required": False
    },
    "secondary sources": {  # SECONDARY_SRC in HTML
        "attributes": ["docid", "uri"],  # title, source, fragment, authors
        "required": False
    }
}


class VerificationResult:
    """Container for verification results with details about failures."""
    
    def __init__(self, passed: bool, error_type: str = None, details: str = None, tokens: list = None):
        self.passed = passed
        self.error_type = error_type  # "hallucination", "consistency", "label_scheme"
        self.details = details
        self.tokens = tokens  # The problematic tokens for debugging
    
    def __bool__(self):
        return self.passed
    
    def __repr__(self):
        if self.passed:
            return "VerificationResult(passed=True)"
        return f"VerificationResult(passed=False, error='{self.error_type}', details='{self.details}')"


def check_hallucination(original_tokens: list, processed_tokens: list, 
                       normalize: bool = True, keep_manual_label: bool = True) -> VerificationResult:
    """
    Verify that processed tokens match original tokens when labels are stripped.
    
    This checks if the LLM hallucinated or modified the original text content
    by comparing cleaned versions of both token lists.
    
    Args:
        original_tokens: Original token list
        processed_tokens: Processed token list with auto_label tags
        normalize: Whether to normalize whitespace during comparison
        keep_manual_label: Whether to keep manual_label tags during comparison
    
    Returns:
        VerificationResult with passed=True if no hallucination detected
    """
    
    original_cleaned = clean_tokens(
        original_tokens, 
        normalize=normalize, 
        keep_manual_label=keep_manual_label,
        keep_auto_label=False,
        keep_bookmarks=False
    )
    
    processed_cleaned = clean_tokens(
        processed_tokens,
        normalize=normalize,
        keep_manual_label=keep_manual_label,
        keep_auto_label=False,
        keep_bookmarks=False
    )
    
    if original_cleaned == processed_cleaned:
        return VerificationResult(passed=True)
    
    # Find where they differ
    diff_details = _find_token_differences(original_cleaned, processed_cleaned)
    
    return VerificationResult(
        passed=False,
        error_type="hallucination",
        details=diff_details,
        tokens=processed_tokens
    )


def check_consistency(tokens: list) -> VerificationResult:
    """
    Verify that all auto_label opening and closing tags are properly balanced.
    
    Checks:
    - Every <auto_label ...> has a matching </auto_label>
    - Tags are properly nested (no cross-nesting)
    
    Args:
        tokens: List of tokens to check
    
    Returns:
        VerificationResult with passed=True if tags are balanced
    """
    # Import inside function to avoid circular imports
    import sys
    import os    
    
    stack = []
    
    for i, token in enumerate(tokens):
        tag_type = is_auto_label_tag(token)
        
        if tag_type == 1:  # Opening tag
            stack.append((token, i))
        elif tag_type == 2:  # Closing tag
            if not stack:
                return VerificationResult(
                    passed=False,
                    error_type="consistency",
                    details=f"Closing tag at position {i} without matching opening tag: {token}",
                    tokens=tokens
                )
            stack.pop()
    
    if stack:
        unclosed = [f"{tok} at position {pos}" for tok, pos in stack]
        return VerificationResult(
            passed=False,
            error_type="consistency",
            details=f"Unclosed tags: {', '.join(unclosed)}",
            tokens=tokens
        )
    
    return VerificationResult(passed=True)


def check_label_scheme(tokens: list, allowed_labels: Optional[List[str]] = None) -> VerificationResult:
    """
    Verify that all label names and attributes conform to the label scheme.
    
    Checks:
    - Label names are in the allowed set (LABEL_SCHEME or custom allowed_labels)
    - Attributes match the expected attributes for each label type
    - No invalid or random label names
    
    Args:
        tokens: List of tokens to check
        allowed_labels: Optional list of allowed label names. If None, uses LABEL_SCHEME keys.
    
    Returns:
        VerificationResult with passed=True if all labels conform to scheme
    """
    # Import inside function to avoid circular imports
    import sys
    import os    
    
    # Determine allowed label names
    if allowed_labels is None:
        allowed_labels = list(LABEL_SCHEME.keys())
    
    # Normalize allowed labels to lowercase for case-insensitive comparison
    allowed_labels_lower = [label.lower() for label in allowed_labels]
    
    invalid_labels = []
    invalid_attributes = []
    
    for i, token in enumerate(tokens):
        if is_auto_label_tag(token) == 1:  # Opening tag only
            try:
                label = HTMLLabel(token)
                label_name = label.name.lower()
                
                # Check if label name is allowed
                if label_name not in allowed_labels_lower:
                    invalid_labels.append(f"'{label.name}' at position {i}")
                    continue
                
                # Check attributes if label is in LABEL_SCHEME
                if label_name in LABEL_SCHEME:
                    expected_attrs = LABEL_SCHEME[label_name]["attributes"]
                    actual_attrs = [k for k in label.attributes.keys() if k != "labelname"]
                    
                    # Find unexpected attributes
                    unexpected = [attr for attr in actual_attrs if attr not in expected_attrs]
                    if unexpected:
                        invalid_attributes.append(
                            f"Label '{label.name}' at position {i} has unexpected attributes: {unexpected}"
                        )
            
            except ValueError as e:
                # Couldn't parse the label
                invalid_labels.append(f"Unparseable label at position {i}: {str(e)}")
    
    # Build result
    if invalid_labels or invalid_attributes:
        details_parts = []
        if invalid_labels:
            details_parts.append(f"Invalid label names: {', '.join(invalid_labels)}")
        if invalid_attributes:
            details_parts.append(f"Invalid attributes: {'; '.join(invalid_attributes)}")
        
        return VerificationResult(
            passed=False,
            error_type="label_scheme",
            details=" | ".join(details_parts),
            tokens=tokens
        )
    
    return VerificationResult(passed=True)


def verify_processed_chunk(original_tokens: list, processed_tokens: list,
                          allowed_labels: Optional[List[str]] = None,
                          check_scheme: bool = True) -> VerificationResult:
    """
    Run all verification checks on a processed chunk.
    
    Performs three checks in order:
    1. Hallucination check (text content unchanged)
    2. Consistency check (tags properly balanced)
    3. Label scheme check (labels and attributes valid)
    
    Returns first failure encountered, or success if all pass.
    
    Args:
        original_tokens: Original token list before processing
        processed_tokens: Processed token list with auto_label tags
        allowed_labels: Optional list of allowed label names
        check_scheme: Whether to check label scheme compliance (default: True)
    
    Returns:
        VerificationResult with details about first failure, or success
    """
    # Check 1: Hallucination
    result = check_hallucination(original_tokens, processed_tokens)
    if not result:
        return result
    
    # Check 2: Consistency
    result = check_consistency(processed_tokens)
    if not result:
        return result
    
    # Check 3: Label scheme (optional)
    if check_scheme:
        result = check_label_scheme(processed_tokens, allowed_labels)
        if not result:
            return result
    
    return VerificationResult(passed=True)


def _find_token_differences(tokens1: list, tokens2: list, context: int = 5) -> str:
    """
    Find where two token lists differ and return a description.
    
    Args:
        tokens1: First token list
        tokens2: Second token list
        context: Number of tokens to show before/after difference
    
    Returns:
        String describing the difference location
    """
    min_len = min(len(tokens1), len(tokens2))
    
    # Find first difference
    for i in range(min_len):
        if tokens1[i] != tokens2[i]:
            start = max(0, i - context)
            end = min(min_len, i + context + 1)
            
            context1 = tokens1[start:end]
            context2 = tokens2[start:end]
            
            return (f"First difference at position {i}:\n"
                   f"  Original: ...{' '.join(context1)}...\n"
                   f"  Processed: ...{' '.join(context2)}...")
    
    # Lists match up to min_len but have different lengths
    if len(tokens1) != len(tokens2):
        return (f"Token lists have different lengths: "
               f"original={len(tokens1)}, processed={len(tokens2)}")
    
    return "No differences found"


#### direct post processing utils

In [22]:
def extract_start_end_tokens(tokens: list) -> list:
    """
    Extract tokens between <start> and <end> markers.
    
    Args:
        tokens: List of tokens containing <start> and <end> markers
    
    Returns:
        List of tokens between markers, or original tokens if markers not found
    
    Raises:
        ValueError: If <start> found but <end> not found, or vice versa
    """
    try:
        start_index = tokens.index("<start>")
    except ValueError:
        print("   ⚠ Warning: <start> marker not found, returning original tokens")
        return tokens
    
    try:
        end_index = tokens.index("<end>")
    except ValueError:
        raise ValueError("<start> marker found but <end> marker missing")
    
    if end_index <= start_index:
        raise ValueError("<end> marker appears before <start> marker")
    
    extracted = tokens[start_index + 1 : end_index]
    print(f"   ✓ Extracted {len(extracted)} tokens between <start> and <end>")
    return extracted


def simplified_to_normal_form(tokens: list, label_type: str = 'auto_label') -> list:
    """
    Convert simplified label format to normal auto_label or manual_label format.
    
    Transforms:
        <decision> → <auto_label labelname="decision">
        </decision> → </auto_label>
        <title titletype="main"> → <auto_label labelname="title" titletype="main">
        </title> → </auto_label>
    
    Args:
        tokens: List of tokens potentially containing simplified label tags
        label_type: Either 'auto_label' or 'manual_label' (default: 'auto_label')
    
    Returns:
        List of tokens with normalized label format
    """
    if label_type not in ['auto_label', 'manual_label']:
        raise ValueError(f"label_type must be 'auto_label' or 'manual_label', got: {label_type}")
    
    normalized_tokens = []
    
    for token in tokens:
        # Check for opening tag: <...> but not </...> or <manual_label...> or <auto_label...>
        is_open = bool(re.fullmatch(r'<(?!\/|manual_label|auto_label)[^>]+>', token))
        
        # Check for closing tag: </...> but not </manual_label...> or </auto_label...>
        is_close = bool(re.fullmatch(r'<\/((?!manual_label|auto_label)[^>]+)>', token))
        
        if is_open:
            # Convert simplified opening tag to normal form
            html_label = from_simplified(token, label_type=label_type)
            normalized_tokens.append(html_label._token)
        elif is_close:
            # Convert simplified closing tag to normal form
            normalized_tokens.append(f'</{label_type}>')
        else:
            # Keep token as-is
            normalized_tokens.append(token)
    
    print(f"   ✓ Converted {len(tokens)} tokens from simplified to {label_type} format")
    return normalized_tokens


def apply_post_processing_transforms(raw_output: str, use_simplified: bool = False, label_type: str = 'auto_label') -> list:
    """
    Apply all post-processing transformations to raw LLM output.
    
    Pipeline:
    1. Tokenize raw output
    2. Extract tokens between <start> and <end> markers
    3. Convert simplified format to normal form (if applicable)
    
    Args:
        raw_output: Raw text output from LLM
        use_simplified: Whether the LLM output uses simplified format
        label_type: Target label type ('auto_label' or 'manual_label')
    
    Returns:
        List of processed tokens ready for verification
    """
    # Step 1: Tokenize
    tokens = tokenize(raw_output)
    print(f"   → Step 1: Tokenized into {len(tokens)} tokens")
    
    # Step 2: Extract between <start> and <end>
    try:
        tokens = extract_start_end_tokens(tokens)
        print(f"   → Step 2: Extracted {len(tokens)} tokens between markers")
    except ValueError as e:
        print(f"   ⚠ Warning: {str(e)}")
    
    # Step 3: Convert simplified to normal form if needed
    if use_simplified:
        tokens = simplified_to_normal_form(tokens, label_type=label_type)
        print(f"   → Step 3: Converted to {label_type} format")
    
    return tokens

#### process chunck

In [23]:
def distance_lists_auto_label(original, derived):
    """
    Calculate the minimum edit distance between two lists and return the operations
    needed to transform the derived list into the original list.
    
    <auto_label> tags can NEVER be modified.
    They can only be deleted. This prevents the algorithm from trying to modify
    auto_label tags to match the original.
    
    Args:
        original: The target list (what we want to achieve)
        derived: The source list (what we start with)
    
    Returns:
        tuple: (distance, operations)
            - distance: minimum number of operations needed
            - operations: list of operations to apply to derived to get original
                         Each operation is a tuple: ('insert', index, value), ('delete', index), or ('modify', index, value)
    
    Example:
        original = ['Act', ',', '\\n', 'S']
        derived = ['Act', '</auto_label>', ',', ' ', '<auto_label labelname="reference">', 'S']
        -> [('delete', 1), ('modify', 2, '\\n'), ('delete', 3)]
    """
    n = len(original)
    m = len(derived)
    

    # Create DP table: dp[i][j] = min operations to transform derived[:j] into original[:i]
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    
    # Base cases
    for i in range(n + 1):
        dp[i][0] = i  # Need i insertions to get from empty list to original[:i]
    for j in range(m + 1):
        dp[0][j] = j  # Need j deletions to get from derived[:j] to empty list
    
    # Fill DP table
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if original[i-1] == derived[j-1]:
                # No operation needed
                dp[i][j] = dp[i-1][j-1]
            else:
                # Calculate costs for each operation
                insert_cost = dp[i-1][j] + 1  # Insert original[i-1] at position j in derived
                delete_cost = dp[i][j-1] + 1  # Delete derived[j-1]
                
                # Modify is only allowed if derived[j-1] is NOT a auto_label tag
                if is_auto_label_tag(derived[j-1]) != 0:
                    # Cannot modify auto_label tags - set modify cost to infinity
                    modify_cost = float('inf')
                else:
                    modify_cost = dp[i-1][j-1] + 1  # Modify derived[j-1] to original[i-1]
                
                dp[i][j] = min(insert_cost, delete_cost, modify_cost)
    
    # Backtrack to find operations
    operations = []
    i, j = n, m
    
    while i > 0 or j > 0:
        if i == 0:
            # Need to delete all remaining items from derived
            operations.append(('delete', j-1))
            j -= 1
        elif j == 0:
            # Need to insert all remaining items from original
            operations.append(('insert', 0, original[i-1]))
            i -= 1
        elif original[i-1] == derived[j-1]:
            # No operation needed, items match
            i -= 1
            j -= 1
        else:
            # Find which operation was taken
            insert_cost = dp[i-1][j]
            delete_cost = dp[i][j-1]
            
            # Check modify cost (will be inf if derived[j-1] is a auto_label)
            if is_auto_label_tag(derived[j-1]) != 0:
                modify_cost = float('inf')
            else:
                modify_cost = dp[i-1][j-1]
            
            min_cost = min(insert_cost, delete_cost, modify_cost)
            
            if min_cost == modify_cost:
                # Modify operation
                operations.append(('modify', j-1, original[i-1]))
                i -= 1
                j -= 1
            elif min_cost == delete_cost:
                # Delete operation
                operations.append(('delete', j-1))
                j -= 1
            else:
                # Insert operation
                operations.append(('insert', j, original[i-1]))
                i -= 1
    
    # Reverse operations since we backtracked
    operations.reverse()
    
    # Adjust indices: operations are relative to the derived list as it's being transformed
    # We need to adjust indices to account for previous operations
    adjusted_operations = []
    offset = 0
    
    for op in operations:
        if op[0] == 'insert':
            adjusted_operations.append(('insert', op[1] + offset, op[2]))
            offset += 1
        elif op[0] == 'delete':
            adjusted_operations.append(('delete', op[1] + offset))
            offset -= 1
        else:  # modify
            adjusted_operations.append(('modify', op[1] + offset, op[2]))
        
    
    distance = dp[n][m]
    return distance, adjusted_operations



def apply_operations_safe(processed_tokens, operations):
    """
    Apply a list of operations to a processed token list, with protection for auto_label tags.
    
    Args:
        processed_tokens: List of tokens to modify
        operations : list of operation

    
    Returns:
        result_tokens: Modified token list with operations applied
    
    Rules:
        - Insert: Always insert at the given index
        - Delete: Delete at index ONLY if it's not a auto_label tag
                 If skipped, adjust all following operation indices by +1
        - Modify: Modify at index ONLY if it's not a auto_label tag
    
    Key insight: When we skip a delete on a auto_label tag, the list doesn't 
    change but the original operation assumed it would. So we need to shift subsequent 
    indices to compensate.
    """
    result_tokens = processed_tokens[:]
    
    # Adjust operations as we go
    adjusted_ops = list(operations)
    
    i = 0
    while i < len(adjusted_ops):
        op = adjusted_ops[i]
        op_type = op[0]
        
        if op_type == 'insert':
            # Always insert
            index = op[1]
            value = op[2]
            result_tokens.insert(index, value)
        
        elif op_type == 'delete':
            # Delete only if not a auto_label tag
            index = op[1]
            if 0 <= index < len(result_tokens):
                if is_auto_label_tag(result_tokens[index]) == 0:
                    result_tokens.pop(index)
                else:
                    # Skipped deletion - adjust all following indices by +1
                    for j in range(i + 1, len(adjusted_ops)):
                        next_op = adjusted_ops[j]

                        if next_op[1] >= index:
                            if next_op[0] == 'insert':
                                adjusted_ops[j] = ('insert', next_op[1] + 1, next_op[2])
                            elif next_op[0] == 'delete':
                                adjusted_ops[j] = ('delete', next_op[1] + 1)
                            elif next_op[0] == 'modify':
                                adjusted_ops[j] = ('modify', next_op[1] + 1, next_op[2])
        
        elif op_type == 'modify':
            # Modify only if not a auto_label tag
            index = op[1]
            value = op[2]
            if 0 <= index < len(result_tokens):
                if is_auto_label_tag(result_tokens[index]) == 0:
                    result_tokens[index] = value
        
        i += 1
    
    return result_tokens

In [24]:
import os
import json
from tqdm import tqdm
from typing import List, Tuple, Optional

class ProcessingHistory:
    """Track processing history for debugging and analysis."""
    
    def __init__(self):
        self.entries = []
    
    def add(self, status: str, chunk_idx: int, raw_output: str, error_details: str = None):
        """Add an entry to the history."""
        self.entries.append({
            "status": status,
            "chunk_idx": chunk_idx,
            "raw_output": raw_output,
            "error_details": error_details
        })
    
    def save(self, output_dir: str, filename: str):
        """Save history to JSON file."""
        if not output_dir or not filename:
            return
        
        json_path = os.path.join(output_dir, f"history_{filename}.json")
        try:
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(self.entries, f, ensure_ascii=False, indent=4)
            print(f"   ✓ Processing history saved to: {json_path}")
        except Exception as e:
            print(f"   ✗ Error saving history: {e}")
    
    def summary(self) -> dict:
        """Get summary statistics."""
        statuses = [entry["status"] for entry in self.entries]
        return {
            "total": len(statuses),
            "success": statuses.count("Success"),
            "hallucination_fail": statuses.count("Hallucination Fail"),
            "consistency_fail": statuses.count("Consistency Fail"),
            "label_scheme_fail": statuses.count("Label Scheme Fail"),
            "double_hallucination_fail": statuses.count("Double Hallucination Fail"),
            "double_consistency_fail": statuses.count("Double Consistency Fail")
        }


def process_single_chunk(
    model,
    chunk: list,
    system_prompt: str,
    user_prompt_template: str,
    label_config: dict,
    allowed_labels: Optional[List[str]] = None,
    max_fallback_attempts: int = 1
) -> Tuple[list, str, str]:
    """
    Process a single chunk with post-processing, verification, and fallback.
    
    Pipeline:
    1. Prepare input (clean chunk)
    2. Generate LLM output
    3. Post-process output (extract, transform)
    4. Verify output (hallucination, consistency, label scheme)
    5. If verification fails, apply error correction and retry
    6. If still fails after max attempts, return original chunk
    
    Args:
        model: LLM model instance
        chunk: List of tokens to process
        system_prompt: System prompt for LLM
        user_prompt_template: User prompt template with {text} placeholder
        label_config: Configuration for label transformations
        allowed_labels: List of allowed label names for scheme validation
        max_fallback_attempts: Maximum number of fallback attempts per error type
    
    Returns:
        Tuple of (processed_tokens, status, error_details)
        - processed_tokens: Successfully processed tokens or original chunk if failed
        - status: "Success", "Hallucination Fail", "Consistency Fail", etc.
        - error_details: Description of error if failed, None if success
    """    
    # ------ 1. PREPARE INPUT ------
    cleaned_chunk = _prepare_label_tokens(
        chunk,
        label_config={
            "keep_attributes": ["labelname"],
            "switch_type": False,
            "use_simplified": False
        }
    )
    text = decode(cleaned_chunk)
    
    # ------ 2. GENERATE LLM OUTPUT ------
    user_prompt = user_prompt_template.format(text=text)
    raw_output = model.generate(
        system_prompt=system_prompt,
        user_prompt=user_prompt
    )
    
    # ------ 3. POST-PROCESS OUTPUT ------
    try:
        processed_tokens = apply_post_processing_transforms(
            raw_output=raw_output,
            use_simplified=label_config.get("use_simplified", False),
            label_type='auto_label'
        )
    except Exception as e:
        return chunk, "Post-processing Error", f"Failed to post-process: {str(e)}"
    
    # ------ 4. APPLY ERROR CORRECTION (BEFORE VERIFICATION) ------
    # This aligns tokens to handle minor discrepancies
    _, operations = distance_lists_auto_label(cleaned_chunk, processed_tokens)
    processed_tokens_corrected = apply_operations_safe(processed_tokens, operations)
    
    # ------ 5. VERIFY OUTPUT ------
    verification = verify_processed_chunk(
        original_tokens=cleaned_chunk,
        processed_tokens=processed_tokens_corrected,
        allowed_labels=allowed_labels,
        check_scheme=True
    )
    
    if verification.passed:
        return processed_tokens_corrected, "Success", None
    
    # ------ 6. HANDLE VERIFICATION FAILURES WITH FALLBACK ------
    print(f"   ⚠ Verification failed: {verification.error_type}")
    print(f"   Details: {verification.details}")
    
    # Determine which fallback to use
    if verification.error_type == "hallucination":
        fallback_system, fallback_user_template = get_prompt_fallback_hallucination()
        failure_type = "Hallucination Fail"
    elif verification.error_type == "consistency":
        fallback_system, fallback_user_template = get_prompt_fallback_consistency()
        failure_type = "Consistency Fail"
    elif verification.error_type == "label_scheme":
        # For label scheme errors, we could use a specialized fallback
        # For now, treat similar to consistency issues
        fallback_system, fallback_user_template = get_prompt_fallback_consistency()
        failure_type = "Label Scheme Fail"
    else:
        return chunk, failure_type, verification.details
    
    # Attempt fallback correction
    for attempt in range(max_fallback_attempts):
        print(f"   → Fallback attempt {attempt + 1}/{max_fallback_attempts}...")
        
        fallback_user_prompt = fallback_user_template.format(
            text_pair=f"Original Text:\n\n<<<ORIGINAL_TEXT>>>{text}<<<END_ORIGINAL_TEXT>>>\n\n"
                     f"Annotated Text with errors:\n\n<<<ANNOTATED_TEXT>>>{raw_output}<<<END_ANNOTATED_TEXT>>>\n\n"
        )
        
        raw_output = model.generate(
            system_prompt=fallback_system,
            user_prompt=fallback_user_prompt
        )
        
        # Post-process fallback output
        try:
            processed_tokens = apply_post_processing_transforms(
                raw_output=raw_output,
                use_simplified=label_config.get("use_simplified", False),
                label_type='auto_label'
            )
        except Exception as e:
            print(f"   ✗ Fallback post-processing failed: {e}")
            continue
        
        # Apply error correction
        _, operations = distance_lists_auto_label(cleaned_chunk, processed_tokens)
        processed_tokens_corrected = apply_operations_safe(processed_tokens, operations)
        
        # Verify again
        verification = verify_processed_chunk(
            original_tokens=cleaned_chunk,
            processed_tokens=processed_tokens_corrected,
            allowed_labels=allowed_labels,
            check_scheme=True
        )
        
        if verification.passed:
            print(f"   ✓ Fallback succeeded on attempt {attempt + 1}")
            return processed_tokens_corrected, f"Success (after {attempt + 1} fallback)", None
    
    # All attempts failed
    print(f"   ✗ All fallback attempts failed")
    return chunk, f"Double {failure_type}", verification.details


def process_chunks(
    model,
    token_chunks: list,
    process_prompt_path: str,
    label_config: dict,
    few_shot_examples: Optional[list] = None,
    allowed_labels: Optional[List[str]] = None,
    output_dir: Optional[str] = None,
    filename: Optional[str] = None,
    max_fallback_attempts: int = 1
) -> list:
    """
    Process multiple chunks using AI model with verification and fallback.
    
    This is the main entry point for chunk processing. It coordinates:
    - LLM generation
    - Post-processing transformations
    - Verification checks
    - Error correction and fallbacks
    - History tracking
    
    Args:
        model: LLM model instance
        token_chunks: List of token chunks to process
        process_prompt_path: Path to the main processing prompt file
        label_config: Configuration for label transformations
        few_shot_examples: Optional list of (input, output) examples
        allowed_labels: Optional list of allowed label names
        output_dir: Optional directory to save outputs
        filename: Optional filename prefix for outputs
        max_fallback_attempts: Maximum fallback attempts per error type
    
    Returns:
        List of processed token chunks
    """
    # Load prompts
    system_prompt, user_prompt_template = get_prompt_processing(
        prompt_path=process_prompt_path,
        few_shot_examples=few_shot_examples
    )
    
    # Initialize tracking
    processed_chunks = []
    history = ProcessingHistory()
    
    print(f"   ✓ Processing {len(token_chunks)} chunks with LLM...")
    print(f"   ✓ Using {len(few_shot_examples) if few_shot_examples else 0} few-shot examples")
    if allowed_labels:
        print(f"   ✓ Label scheme validation enabled with {len(allowed_labels)} allowed labels")
    
    # Process each chunk
    for idx, chunk in enumerate(tqdm(token_chunks, desc="Processing chunks")):
        processed_tokens, status, error_details = process_single_chunk(
            model=model,
            chunk=chunk,
            system_prompt=system_prompt,
            user_prompt_template=user_prompt_template,
            label_config=label_config,
            allowed_labels=allowed_labels,
            max_fallback_attempts=max_fallback_attempts
        )
        
        processed_chunks.append(processed_tokens)
        history.add(status, idx, decode(processed_tokens), error_details)
        
        if status != "Success" and not status.startswith("Success (after"):
            print(f"   ⚠ Chunk {idx} failed: {status}")
    
    # Save history and results
    history.save(output_dir, filename)
    
    # Print summary
    summary = history.summary()
    print(f"\n   ✓ Processing completed:")
    print(f"      - Total chunks: {summary['total']}")
    print(f"      - Successful: {summary['success']}")
    print(f"      - Failed: {summary['total'] - summary['success']}")
    
    if output_dir and filename:
        json_path = os.path.join(output_dir, f"processed_chunks_{filename}.json")
        try:
            # Convert token lists to strings for JSON serialization
            chunks_as_strings = [decode(chunk) for chunk in processed_chunks]
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(chunks_as_strings, f, indent=4, ensure_ascii=False)
            print(f"   ✓ Processed chunks saved to: {json_path}")
        except Exception as e:
            print(f"   ✗ Error saving processed chunks: {e}")
    
    return processed_chunks

#### main

In [25]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
model_name = "gpt-4.1"

n_few_shot = 10  # Number of few-shot examples to use

In [26]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code"
filename = "1999CanLII7320_annotated"
anno = "VP"
version = "v1"
html_path = fr"{project_root}\labelstudio\annotation\data\test\1999CanLII7320_annotated_VP_v1.html"
output_dir = fr"{project_root}\labelstudio\annotation\data\test\test_llm"
os.makedirs(output_dir, exist_ok=True)

# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")

   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\labelstudio\annotation\data\test\1999CanLII7320_annotated_VP_v1.html


In [27]:
# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Extract body content ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)

   ✓ Found bookmark separator at index 23044
   ✓ Splitting: 23044 tokens before, 31001 tokens after
   ✓ Chunked tokens into 34 chunks (>= 500 tokens each)
   ✓ Chunked tokens into 62 chunks (>= 500 tokens each)
   ✓ Total chunks: 34 before + 62 after = 96


In [28]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
}

In [29]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                              label_config)



selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")


   ✓ Extracted 34 few-shot examples from chunks
   ✓ Selected 10 few-shot examples for processing.


In [30]:
system_prompt, user_prompt_template = get_prompt_processing(
        prompt_path=fr"{project_root}\labelstudio\annotation\llm_based_annotation\utils\prompts\simplified_parent_extraction_cot.txt",
        few_shot_examples=few_shot_examples
    )

In [85]:
print(system_prompt)

You are an advanced Language Model designed to annotate Canadian legal texts by identifying and marking **mentions of legal authorities** in judicial decisions.

You must follow the instructions below **with extreme care**. Pay **very close attention to the provided examples**, as they precisely demonstrate **what should be annotated, how spans should be selected, and how labels should be applied**. The examples are not illustrative only — they define the expected behavior.

---

## Task Objective

Identify and annotate **every mention of a legal authority** appearing in the text, whether the reference is **complete or partial**, using the specified XML schema.

**Important restriction**
Do **NOT** annotate:

* Names of parties (plaintiffs, defendants, applicants, respondents)
* Names of individuals, companies, or organizations **unless they are part of a cited legal authority**

Your task is **strictly limited** to identifying **document sources**, not stakeholders or participants in 

In [31]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name)

In [32]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\labelstudio\annotation\llm_based_annotation\utils\prompts\simplified_parent_extraction_cot.txt"

processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunk2,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename
)

   ✓ Processing 62 chunks with LLM...
   ✓ Using 10 few-shot examples


Processing chunks:   2%|▏         | 1/62 [00:04<04:43,  4.64s/it]

   → Step 1: Tokenized into 504 tokens
   ✓ Extracted 502 tokens between <start> and <end>
   → Step 2: Extracted 502 tokens between markers
   ✓ Converted 502 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing chunks:   3%|▎         | 2/62 [00:10<05:18,  5.31s/it]

   → Step 1: Tokenized into 503 tokens
   ✓ Extracted 501 tokens between <start> and <end>
   → Step 2: Extracted 501 tokens between markers
   ✓ Converted 501 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


...

   → Step 1: Tokenized into 509 tokens
   ✓ Extracted 507 tokens between <start> and <end>
   → Step 2: Extracted 507 tokens between markers
   ✓ Converted 507 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing chunks:  98%|█████████▊| 61/62 [04:35<00:03,  3.44s/it]

   → Step 1: Tokenized into 505 tokens
   ✓ Extracted 503 tokens between <start> and <end>
   → Step 2: Extracted 503 tokens between markers
   ✓ Converted 503 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing chunks: 100%|██████████| 62/62 [04:39<00:00,  4.51s/it]

   → Step 1: Tokenized into 472 tokens
   ✓ Extracted 470 tokens between <start> and <end>
   → Step 2: Extracted 470 tokens between markers
   ✓ Converted 470 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format
   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\labelstudio\annotation\data\test\test_llm\history_1999CanLII7320_annotated.json

   ✓ Processing completed:
      - Total chunks: 62
      - Successful: 62
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\labelstudio\annotation\data\test\test_llm\processed_chunks_1999CanLII7320_annotated.json


#### Post processing

In [39]:
# ---------- Merge all tokens ----------
body_chunked_processed =  token_chunk1 + processed_chunks
processed_tokens_flat = flatten_token_chunks(body_chunked_processed)


original_tokens = tokenize(html_content)
processed_html_content = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)

processed_html = decode(processed_html_content)

print(f"\nMerged HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
processed_html_content = add_style_and_parent_to_auto_labels(processed_html)

   ✓ Flattened 96 chunks into 54283 tokens

Merged HTML length: 247112


In [40]:
# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{anno}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\labelstudio\annotation\data\test\test_llm
